# **YouTube Videolarını OpenCV'ye Aktarın**

#### **Bu derste şunları öğreneceğiz:**
1. Opencv'de YouTube Videolarını içe aktarmak için Pafy Kütüphanesi nasıl kullanılır?
2. Youtube videolarından META verileri nasıl elde edilir
3. Videoyu veya sesi bir youtube bağlantısından indirin

**Yüklemeniz gerekenler:**
1. pip install yt-dlp


In [ ]:
pip install yt-dlp

## OpenCV'de bir YouTube Videosu Görüntüleme

In [ ]:
import cv2
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A?si=Hvur2sRyNOoYBjiR"

ydl_opts = {"format": "best", "quiet": True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)
    stream_url = info["url"]

cap = cv2.VideoCapture(stream_url)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("YouTube", frame)
    if cv2.waitKey(1) == 13:  # Enter tuşu
        break

cap.release()
cv2.destroyAllWindows()


### Video META Verilerini Al

In [ ]:
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A?si=Hvur2sRyNOoYBjiR"

ydl_opts = {"quiet": True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

print("Title: {}".format(info.get("title")))
print("Rating: {}".format(info.get("average_rating")))  # bazen None olabilir
print("Viewcount: {}".format(info.get("view_count")))
print("Author: {}".format(info.get("uploader")))
print("Length: {} seconds".format(info.get("duration")))
print("Duration: {}".format(info.get("duration_string")))


### Mevcut Akışları Görüntüleyelim

In [ ]:
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A?si=Hvur2sRyNOoYBjiR"

ydl_opts = {"quiet": True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

# "formats" içinde tüm stream seçenekleri var
for f in info.get("formats", []):
    resolution = f.get("resolution") or f"{f.get('width')}x{f.get('height')}"
    extension = f.get("ext")
    filesize = f.get("filesize") or f.get("filesize_approx")
    stream_url = f.get("url")

    print("Resolution:", resolution)
    print("Extension:", extension)
    print("Filesize:", filesize)
    print("URL:", stream_url)
    print("-" * 40)

### En yüksek kalitede akışı elde edelim

In [ ]:
import yt_dlp

url = "https://youtu.be/EFEmTsfFL5A"

# "best" format -> video+audio birlikte (varsa), yoksa en iyi tek akış
ydl_opts = {"format": "best", "quiet": True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

# Seçilen en iyi format bilgisi
resolution = f"{info.get('width')}x{info.get('height')}" if info.get("width") else "audio only"
extension = info.get("ext")
stream_url = info.get("url")

print("Resolution:", resolution)
print("Extension:", extension)
print("URL:", stream_url)


### Videoları İndir 

In [ ]:
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A"

ydl_opts = {
    "format": "bestvideo+bestaudio/best",  # önce ayrı video+ses birleştir, olmazsa tek best
    "outtmpl": "%(title)s.%(ext)s",        # çıktı dosya adı
    "merge_output_format": "mp4",          # çıktı formatı
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

### Ses Al & İndir 

In [ ]:
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A"

ydl_opts = {"quiet": True}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

# Tüm ses akışlarını filtrele
audiostreams = [
    f for f in info.get("formats", [])
    if f.get("vcodec") == "none"   # sadece ses (video codec yok)
]

for a in audiostreams:
    bitrate = a.get("abr") or a.get("tbr")   # abr = audio bitrate (kbps)
    extension = a.get("ext")
    filesize = a.get("filesize") or a.get("filesize_approx")
    url = a.get("url")

    print("Bitrate:", bitrate, "kbps")
    print("Extension:", extension)
    print("Filesize:", filesize)
    print("URL:", url)
    print("-" * 40)


In [ ]:
import yt_dlp

url = "https://youtu.be/1XiIlZ74m1A"

# Örneğin format_id = "140" (m4a 128 kbps) gibi belirli bir ses stream'i
format_id = "140"

ydl_opts = {
    "format": format_id,                  # seçtiğin ses stream'i
    "outtmpl": "%(title)s.%(ext)s",       # önce orijinal formatta indir
    "postprocessors": [{                  # sonra MP3'e dönüştür
        "key": "FFmpegExtractAudio",
        "preferredcodec": "mp3",
        "preferredquality": "192",        # kbps değeri (128, 192, 256 vs.)
    }],
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])